# College Data Intelligence – Build a Small Language Model (SLM) with Hugging Face

**Beginner-friendly Google Colab project**

This notebook builds a small, fine-tuned text-to-text model using the uploaded college-program dataset. It also adds semantic retrieval so the assistant can find the relevant record before answering.

**Dataset:** `dataset(1).json` (999 records; 72 fields)

**Model:** `google/flan-t5-small`  
**Retrieval model:** `sentence-transformers/all-MiniLM-L6-v2`

> Important: the JSON file is structured tabular data, not a ready-made chat dataset. This notebook creates question-answer training examples from available fields. It splits by institution identifier to reduce institution-level leakage. A separate test set is kept for validation. The language model is a small general pretrained model that we fine-tune; it is not trained from random weights.

**Recommended:** Google Colab → Runtime → Change runtime type → T4 GPU (if available).

## Step 1 – Set up the environment

Run this cell first. It installs the libraries used for data handling, fine-tuning, embeddings, and evaluation.

In [4]:

!pip install -q pyarrow==26.0.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [5]:

import numpy as np
import pandas as pd
import pyarrow
import torch
import transformers
import datasets
import sentence_transformers

print("===== ENVIRONMENT CHECK =====")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

print("\nGPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

print("\nALL IMPORTS SUCCESSFUL!")


===== ENVIRONMENT CHECK =====
NumPy: 2.4.6
Pandas: 2.2.3
PyArrow: 26.0.1
Transformers: 5.22.2
Datasets: 5.2.0

GPU Available: True
GPU Name: Tesla T4

ALL IMPORTS SUCCESSFUL!


## Step 2 – Upload your JSON dataset

When the file picker appears, select the `dataset(1).json` file you uploaded in ChatGPT.

If you have already uploaded it to the Colab session, you can skip the upload cell and set `DATA_PATH` to its location.

In [4]:

from google.colab import files
import os

# Upload your original dataset from your computer
uploaded = files.upload()

# Get the uploaded filename
filename = next(iter(uploaded.keys()))

# Complete Google Colab file path
DATA_PATH = "/content/" + filename

print("Dataset uploaded successfully!")
print("File name:", filename)
print("Dataset path:", DATA_PATH)
print("File exists:", os.path.exists(DATA_PATH))


Saving dataset.json to dataset.json
Dataset uploaded successfully!
File name: dataset.json
Dataset path: /content/dataset.json
File exists: True


## Step 3 – Load and inspect the dataset

This checks the file, number of records, columns, and missing values. The JSON is expected to be a list of objects.

In [5]:

import json
import pandas as pd
import numpy as np

# Load your uploaded JSON dataset
with open("/content/dataset.json", "r", encoding="utf-8") as f:
    records = json.load(f)

# Convert JSON records into a DataFrame
df = pd.DataFrame(records)

# Display dataset information
print("===== DATASET INFORMATION =====")
print("Total Rows:", len(df))
print("Total Columns:", len(df.columns))

print("\n===== FIRST 5 RECORDS =====")
display(df.head())

print("\n===== COLUMN NAMES =====")
print(df.columns.tolist())

print("\n===== MISSING VALUES =====")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nDATASET LOADED SUCCESSFULLY!")


===== DATASET INFORMATION =====
Total Rows: 999
Total Columns: 72

===== FIRST 5 RECORDS =====


,program_id,program_id.1,program_id.2,program_id.3,program_id.4,program_id.5,program_id.6,program_id.7,program_id.8,program_id.9,...,occupations_using_ai_software,ai_software_occupation_share,expert_system_occupation_share,ai_tools_max_per_occupation,hot_technologies_mean_per_occupation,ai_tool_examples,earnings_4yr_status,earnings_1yr_status,earnings_5yr_status,debt_status
0,opeid031505-51.07-1,NaN,31505,A - Technical College,"Private, for-profit",1,NaN,NaN,NaN,NaN,...,1,0.053,0.368,1.0,31.95,OpenAI ChatGPT,reported,reported,reported,reported
1,opeid031505-51.35-1,NaN,31505,A - Technical College,"Private, for-profit",1,NaN,NaN,NaN,NaN,...,0,0.000,0.000,0.0,5.50,NaN,privacy_suppressed,privacy_suppressed,privacy_suppressed,privacy_suppressed
2,opeid031505-52.04-1,NaN,31505,A - Technical College,"Private, for-profit",1,NaN,NaN,NaN,NaN,...,0,0.000,0.167,0.0,18.11,NaN,reported,privacy_suppressed,privacy_suppressed,privacy_suppressed
3,493868-12.04-1,493868.0,42833,A Better U Beauty Barber Academy,"Private, for-profit",1,Albuquerque,NM,Southwest,35.110630,...,0,0.000,0.000,0.0,5.12,NaN,privacy_suppressed,privacy_suppressed,privacy_suppressed,privacy_suppressed
4,177834-13.06-5,177834.0,2477,A T Still University of Health Sciences,"Private, nonprofit",1,Kirksville,MO,Plains,40.193648,...,2,0.333,0.333,2.0,35.00,Angoss KnowledgeSEEKER; SAS Enterprise Miner; ...,privacy_suppressed,not_available,not_available,not_available



===== COLUMN NAMES =====
['program_id', 'program_id.1', 'program_id.2', 'program_id.3', 'program_id.4', 'program_id.5', 'program_id.6', 'program_id.7', 'program_id.8', 'program_id.9', 'program_id.10', 'program_id.11', 'program_id.12', 'program_id.13', 'program_id.14', 'institution_tuition_in_state_usd', 'institution_tuition_out_state_usd', 'cip_code_4digit', 'cip_title', 'cip_family_code', 'cip_family_title', 'credential_level', 'credential_name', 'distance_education', 'awards_year1', 'awards_year2', 'outcomes_shared_across_campuses', 'median_earnings_4yr_usd', 'earnings_cohort_size_4yr', 'national_median_earnings_4yr_usd', 'national_p25_earnings_4yr_usd', 'national_p75_earnings_4yr_usd', 'earnings_vs_national_pct', 'median_earnings_1yr_usd', 'earnings_cohort_size_1yr', 'median_earnings_5yr_usd', 'earnings_cohort_size_5yr', 'not_working_count_5yr', 'count_above_hs_threshold_5yr', 'count_working_in_state_5yr', 'median_debt_usd', 'debt_borrower_count', 'median_monthly_payment_usd', 'ear

,0
earnings_growth_pct_1yr_to_5yr,811
earnings_trajectory_category,811
pct_above_hs_threshold_5yr,811
count_above_hs_threshold_5yr,811
count_working_in_state_5yr,790
pct_working_in_state_5yr,790
debt_to_earnings_1yr,777
payment_to_income_pct_1yr,777
debt_to_earnings_4yr,770
pct_working_5yr,760



DATASET LOADED SUCCESSFULLY!


## Step 4 – Clean values and prepare useful text fields

Missing values are kept as missing rather than being replaced with invented facts. The `program_id.2` field is used as the institution grouping key when available; otherwise, institution name is used.

In [6]:

import pandas as pd
import numpy as np

# Handle missing values
def is_missing(x):
    try:
        return pd.isna(x)
    except Exception:
        return x is None

# Clean individual values
def clean_value(x):
    if is_missing(x):
        return None

    if isinstance(x, (np.integer,)):
        return int(x)

    if isinstance(x, (np.floating, float)):
        if not np.isfinite(x):
            return None
        return float(x)

    return str(x).strip()

# Create institution groups
def get_group(row):
    inst_id = clean_value(row.get("program_id.2"))

    if inst_id is not None:
        return str(inst_id)

    name = clean_value(row.get("program_id.3"))

    if name:
        return str(name)

    return str(row.get("program_id", "unknown"))

df["group_id"] = df.apply(get_group, axis=1)

print("===== DATA CLEANING COMPLETED =====")
print("Total records:", len(df))
print("Unique institutions:", df["group_id"].nunique())
print("Missing values:", df.isnull().sum().sum())

print("\nExample institution:")
print(df.iloc[0]["program_id.3"])

print("\nExample program:")
print(df.iloc[0]["cip_title"])

print("\nData preparation successful!")


===== DATA CLEANING COMPLETED =====
Total records: 999
Unique institutions: 103
Missing values: 20941

Example institution:
A - Technical College

Example program:
Health and Medical Administrative Services

Data preparation successful!


## Step 5 – Create supervised question-answer examples

A language model learns from input/target examples. We generate factual questions from fields that actually exist in each row.

Examples include:
- Which college offers this program?
- What credential does this program award?
- What is the in-state tuition?
- What are the reported four-year median earnings?
- What is the median student debt?
- Which occupation is most strongly linked to the program?

Rows with missing values for a target field are skipped for that particular question.

In [7]:

import pandas as pd
import numpy as np

# Function to safely read values
def get_value(row, column):
    value = row.get(column)

    if pd.isna(value):
        return None

    return value

# Function to format numbers as currency
def format_money(value):
    if value is None:
        return None
    return f"${float(value):,.0f}"

# Generate Question-Answer examples
def build_qa_examples(data):

    examples = []

    for idx, row in data.iterrows():

        institution = get_value(row, "program_id.3")
        program = get_value(row, "cip_title")
        credential = get_value(row, "credential_name")

        # Create context from actual dataset information
        context_parts = []

        fields = {
            "Institution": "program_id.3",
            "Program": "cip_title",
            "Credential": "credential_name",
            "Institution Type": "program_id.4",
            "City": "program_id.6",
            "State": "program_id.7",
        }

        for label, column in fields.items():
            value = get_value(row, column)

            if value is not None:
                context_parts.append(f"{label}: {value}")

        context = " | ".join(context_parts)

        # Question and answer definitions
        questions = [
            (
                "Which institution offers this program?",
                institution
            ),
            (
                "What is the name of this program?",
                program
            ),
            (
                "What credential does this program offer?",
                credential
            ),
            (
                "What is the in-state tuition?",
                format_money(get_value(row, "institution_tuition_in_state_usd"))
            ),
            (
                "What is the out-of-state tuition?",
                format_money(get_value(row, "institution_tuition_out_state_usd"))
            ),
            (
                "What are the median four-year earnings?",
                format_money(get_value(row, "median_earnings_4yr_usd"))
            ),
            (
                "What is the median student debt?",
                format_money(get_value(row, "median_debt_usd"))
            ),
            (
                "Which occupation is most strongly linked?",
                get_value(row, "largest_linked_occupation")
            ),
            (
                "What is the linked occupation's median wage?",
                format_money(get_value(row, "occupation_median_wage_2024_usd"))
            ),
            (
                "What is the earnings trajectory category?",
                get_value(row, "earnings_trajectory_category")
            ),
            (
                "Which AI tools are listed?",
                get_value(row, "ai_tool_examples")
            )
        ]

        for question, answer in questions:

            # Skip missing answers
            if answer is None:
                continue

            prompt = (
                "Answer using only the provided context. "
                "Do not invent information.\n"
                f"Context: {context}\n"
                f"Question: {question}\n"
                "Answer:"
            )

            examples.append({
                "row_index": int(idx),
                "group_id": str(row["group_id"]),
                "question": question,
                "input_text": prompt,
                "target_text": str(answer)
            })

    return pd.DataFrame(examples)


# Generate the training examples
qa_df = build_qa_examples(df)

# Display results
print("===== QUESTION ANSWER DATASET =====")
print("Total QA examples:", len(qa_df))

print("\n===== FIRST 10 EXAMPLES =====")

display(
    qa_df[
        ["question", "input_text", "target_text"]
    ].head(10)
)

print("\nQA DATA GENERATION COMPLETED!")


===== QUESTION ANSWER DATASET =====
Total QA examples: 7282

===== FIRST 10 EXAMPLES =====


,question,input_text,target_text
0,Which institution offers this program?,Answer using only the provided context. Do not...,A - Technical College
1,What is the name of this program?,Answer using only the provided context. Do not...,Health and Medical Administrative Services
2,What credential does this program offer?,Answer using only the provided context. Do not...,Undergraduate Certificate or Diploma
3,What are the median four-year earnings?,Answer using only the provided context. Do not...,"$33,231"
4,What is the median student debt?,Answer using only the provided context. Do not...,"$9,087"
5,Which occupation is most strongly linked?,Answer using only the provided context. Do not...,First-line supervisors of office and administr...
6,What is the linked occupation's median wage?,Answer using only the provided context. Do not...,"$82,269"
7,What is the earnings trajectory category?,Answer using only the provided context. Do not...,steady
8,Which AI tools are listed?,Answer using only the provided context. Do not...,OpenAI ChatGPT
9,Which institution offers this program?,Answer using only the provided context. Do not...,A - Technical College



QA DATA GENERATION COMPLETED!


## Step 6 – Split into training, validation, and testing sets

We split by institution group, not by individual question. This helps prevent the same institution appearing in both training and test data.

Target split: approximately 80% training, 10% validation, and 10% testing. Exact percentages can vary because whole institutions are kept together.

In [8]:

from sklearn.model_selection import GroupShuffleSplit

# Step 1: Prepare institution groups
row_meta = df[["group_id"]].copy()
row_meta["row_index"] = df.index

# Step 2: Separate 10% of institutions for testing
split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

trainval_idx, test_idx = next(
    split1.split(row_meta, groups=row_meta["group_id"])
)

# Step 3: Separate validation data from remaining institutions
trainval_meta = row_meta.iloc[trainval_idx].copy()

split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=1/9,
    random_state=42
)

train_rel, val_rel = next(
    split2.split(
        trainval_meta,
        groups=trainval_meta["group_id"]
    )
)

# Step 4: Get row indices
train_rows = set(
    trainval_meta.iloc[train_rel]["row_index"]
)

val_rows = set(
    trainval_meta.iloc[val_rel]["row_index"]
)

test_rows = set(
    row_meta.iloc[test_idx]["row_index"]
)

# Step 5: Create training, validation and testing datasets
train_df = qa_df[
    qa_df["row_index"].isin(train_rows)
].reset_index(drop=True)

val_df = qa_df[
    qa_df["row_index"].isin(val_rows)
].reset_index(drop=True)

test_df = qa_df[
    qa_df["row_index"].isin(test_rows)
].reset_index(drop=True)

# Step 6: Display dataset sizes
print("===== DATASET SPLIT RESULTS =====")

print("Total QA examples:", len(qa_df))
print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Testing examples:", len(test_df))

# Step 7: Check institution leakage
train_groups = set(train_df["group_id"])
val_groups = set(val_df["group_id"])
test_groups = set(test_df["group_id"])

print("\n===== LEAKAGE CHECK =====")

print("Train/Validation overlap:",
      len(train_groups & val_groups))

print("Train/Test overlap:",
      len(train_groups & test_groups))

print("Validation/Test overlap:",
      len(val_groups & test_groups))

assert len(train_groups & val_groups) == 0
assert len(train_groups & test_groups) == 0
assert len(val_groups & test_groups) == 0

print("\nDATA SPLITTING COMPLETED SUCCESSFULLY!")


===== DATASET SPLIT RESULTS =====
Total QA examples: 7282
Training examples: 5786
Validation examples: 416
Testing examples: 1080

===== LEAKAGE CHECK =====
Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0

DATA SPLITTING COMPLETED SUCCESSFULLY!


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

row_meta = df[["group_id"]].copy()
row_meta["row_index"] = row_meta.index

# First hold out 10% of institution groups for final testing.
split1 = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
trainval_idx, test_idx = next(split1.split(row_meta, groups=row_meta["group_id"]))

trainval_meta = row_meta.iloc[trainval_idx].copy()
# Hold out 1/9 of the remaining groups, approximately 10% of all data, for validation.
split2 = GroupShuffleSplit(n_splits=1, test_size=1/9, random_state=42)
train_rel, val_rel = next(split2.split(trainval_meta, groups=trainval_meta["group_id"]))

train_rows = set(trainval_meta.iloc[train_rel]["row_index"].tolist())
val_rows = set(trainval_meta.iloc[val_rel]["row_index"].tolist())
test_rows = set(row_meta.iloc[test_idx]["row_index"].tolist())

train_df = qa_df[qa_df["row_index"].isin(train_rows)].reset_index(drop=True)
val_df = qa_df[qa_df["row_index"].isin(val_rows)].reset_index(drop=True)
test_df = qa_df[qa_df["row_index"].isin(test_rows)].reset_index(drop=True)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Testing examples:", len(test_df))
print("Institution overlap train/test:", len(set(df.loc[list(train_rows), "group_id"]) & set(df.loc[list(test_rows), "group_id"])))
assert len(train_df) and len(val_df) and len(test_df), "A split is empty. Check the number of institution groups."

## Step 7 – Load a small Hugging Face model

We use `google/flan-t5-small`, a compact instruction-following text-to-text model. It is suitable for a first Colab experiment and much lighter than large chat models.

This is the base model before fine-tuning.

In [9]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

print("Loading FLAN-T5 model...")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("\n===== MODEL INFORMATION =====")
print("Model:", MODEL_NAME)
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nMODEL LOADED SUCCESSFULLY!")


Loading tokenizer...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


===== MODEL INFORMATION =====
Model: google/flan-t5-small
Device: cuda
Parameters: 76961152
GPU Available: True
GPU: Tesla T4

MODEL LOADED SUCCESSFULLY!


## Step 8 – Tokenize the data for training

Tokenization converts text into numerical tokens the model can process. We keep the input length bounded to fit Colab memory.

In [10]:

from datasets import Dataset

# Maximum sequence lengths
MAX_INPUT_LENGTH = 384
MAX_TARGET_LENGTH = 64

# Convert text into tokens
def tokenize_batch(batch):

    inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]

    return inputs


# Convert Pandas DataFrames into Hugging Face datasets
train_ds = Dataset.from_pandas(
    train_df[["input_text", "target_text"]],
    preserve_index=False
)

val_ds = Dataset.from_pandas(
    val_df[["input_text", "target_text"]],
    preserve_index=False
)

test_ds = Dataset.from_pandas(
    test_df[["input_text", "target_text"]],
    preserve_index=False
)


# Tokenize all datasets
train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_ds.column_names
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_ds.column_names
)

test_tok = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=test_ds.column_names
)


# Display results
print("===== TOKENIZATION RESULTS =====")

print("Training samples:", len(train_tok))
print("Validation samples:", len(val_tok))
print("Testing samples:", len(test_tok))

print("\nExample token IDs:")
print(train_tok[0]["input_ids"][:30])

print("\nExample target IDs:")
print(train_tok[0]["labels"])

print("\nTOKENIZATION COMPLETED SUCCESSFULLY!")


Map:   0%|          | 0/5786 [00:00<?, ? examples/s]

Map:   0%|          | 0/416 [00:00<?, ? examples/s]

Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

===== TOKENIZATION RESULTS =====
Training samples: 5786
Validation samples: 416
Testing samples: 1080

Example token IDs:
[11801, 338, 163, 8, 937, 2625, 5, 531, 59, 16, 2169, 251, 5, 1193, 6327, 10, 14932, 10, 71, 3, 18, 10728, 1888, 1820, 2350, 10, 1685, 11, 3721, 25290]

Example target IDs:
[71, 3, 18, 10728, 1888, 1]

TOKENIZATION COMPLETED SUCCESSFULLY!


## Step 9 – Fine-tune (train) the SLM

This is the actual model-training step. The model learns to answer questions using the supplied record context.

If Colab runs out of memory:
1. Change `BATCH_SIZE` from 8 to 4 or 2.
2. Reduce `MAX_INPUT_LENGTH` to 256.
3. Make sure a GPU runtime is selected.

In [11]:

import torch
import inspect

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Prepare data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

# Training configuration
BATCH_SIZE = 8 if torch.cuda.is_available() else 2

# Handle Transformers API version differences
args_params = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

eval_key = (
    "eval_strategy"
    if "eval_strategy" in args_params
    else "evaluation_strategy"
)

training_config = {
    "output_dir": "/content/college_slm_checkpoints",
    eval_key: "epoch",
    "save_strategy": "epoch",
    "learning_rate": 3e-4,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "weight_decay": 0.01,
    "num_train_epochs": 3,
    "logging_steps": 50,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
    "seed": 42
}

if "predict_with_generate" in args_params:
    training_config["predict_with_generate"] = True

training_args = Seq2SeqTrainingArguments(**training_config)

# Prepare trainer
trainer_params = inspect.signature(
    Seq2SeqTrainer.__init__
).parameters

trainer_config = {
    "model": model,
    "args": training_args,
    "train_dataset": train_tok,
    "eval_dataset": val_tok,
    "data_collator": data_collator
}

if "processing_class" in trainer_params:
    trainer_config["processing_class"] = tokenizer
else:
    trainer_config["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_config)

print("===== TRAINING CONFIGURATION =====")
print("Model: FLAN-T5-small")
print("Training examples:", len(train_tok))
print("Validation examples:", len(val_tok))
print("Epochs: 3")
print("Batch size:", BATCH_SIZE)
print("Learning rate: 0.0003")
print("GPU:", torch.cuda.is_available())

print("\nStarting SLM training...")

# Start training
train_result = trainer.train()

print("\nSLM TRAINING COMPLETED SUCCESSFULLY!")


===== TRAINING CONFIGURATION =====
Model: FLAN-T5-small
Training examples: 5786
Validation examples: 416
Epochs: 3
Batch size: 8
Learning rate: 0.0003
GPU: True

Starting SLM training...


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


SLM TRAINING COMPLETED SUCCESSFULLY!


In [12]:

import torch
import math

print("===== MODEL DIAGNOSTICS =====")

# Check model parameters
all_finite = all(
    torch.isfinite(p).all().item()
    for p in model.parameters()
)

print("Model weights finite:", all_finite)

# Check training labels
print("Training examples:", len(train_tok))
print("Validation examples:", len(val_tok))

print("\nExample training labels:")
print(train_tok[0]["labels"])

# Prepare a small training batch
batch = data_collator([
    train_tok[i] for i in range(4)
])

batch = {
    k: v.to(model.device)
    for k, v in batch.items()
}

# Count valid labels
valid_labels = (batch["labels"] != -100).sum().item()

print("\nValid target tokens:", valid_labels)

# Check actual model loss
model.eval()

with torch.no_grad():
    output = model(**batch)

print("Test batch loss:", output.loss.item())
print("Loss is finite:", math.isfinite(output.loss.item()))

print("\n===== DIAGNOSTICS COMPLETED =====")


===== MODEL DIAGNOSTICS =====
Model weights finite: True
Training examples: 5786
Validation examples: 416

Example training labels:
[71, 3, 18, 10728, 1888, 1]

Valid target tokens: 23
Test batch loss: nan
Loss is finite: False

===== DIAGNOSTICS COMPLETED =====


In [13]:

import torch
import math
from transformers import AutoModelForSeq2SeqLM

# Clear previous GPU memory
torch.cuda.empty_cache()

# Reload original pretrained model
MODEL_NAME = "google/flan-t5-small"

print("Reloading original FLAN-T5 model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model = model.to("cuda" if torch.cuda.is_available() else "cpu")

# Prepare a small batch
model.eval()

batch = data_collator([
    train_tok[i] for i in range(4)
])

batch = {
    key: value.to(model.device)
    for key, value in batch.items()
}

# Test loss without mixed precision
with torch.no_grad():
    output = model(**batch)

print("\n===== FRESH MODEL TEST =====")
print("Model dtype:", next(model.parameters()).dtype)
print("Test loss:", output.loss.item())
print("Loss is finite:", math.isfinite(output.loss.item()))

if torch.isfinite(output.loss):
    print("\nFRESH MODEL PASSED!")
else:
    print("\nMODEL STILL HAS A NUMERICAL ISSUE!")


Reloading original FLAN-T5 model...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



===== FRESH MODEL TEST =====
Model dtype: torch.float32
Test loss: 0.8935898542404175
Loss is finite: True

FRESH MODEL PASSED!


In [14]:

import torch
import inspect

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Clear old GPU memory
torch.cuda.empty_cache()

# Use full precision for stable training
model = model.float()
model.config.use_cache = False

# Prepare data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

# Detect Transformers version API
args_params = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

eval_key = (
    "eval_strategy"
    if "eval_strategy" in args_params
    else "evaluation_strategy"
)

# Corrected training settings
training_config = {
    "output_dir": "/content/college_slm_corrected",
    eval_key: "epoch",
    "save_strategy": "epoch",
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": 3,
    "weight_decay": 0.01,
    "logging_steps": 50,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "fp16": False,
    "report_to": "none",
    "seed": 42
}

if "predict_with_generate" in args_params:
    training_config["predict_with_generate"] = False

training_args = Seq2SeqTrainingArguments(
    **training_config
)

# Create the trainer
trainer_params = inspect.signature(
    Seq2SeqTrainer.__init__
).parameters

trainer_config = {
    "model": model,
    "args": training_args,
    "train_dataset": train_tok,
    "eval_dataset": val_tok,
    "data_collator": data_collator
}

if "processing_class" in trainer_params:
    trainer_config["processing_class"] = tokenizer
else:
    trainer_config["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_config)

print("===== CORRECTED TRAINING CONFIGURATION =====")
print("Model: FLAN-T5-small")
print("Training examples:", len(train_tok))
print("Validation examples:", len(val_tok))
print("Epochs: 3")
print("Batch size: 4")
print("Gradient accumulation: 2")
print("Learning rate: 0.00005")
print("FP16: False")
print("GPU:", torch.cuda.get_device_name(0))

# Start corrected training
print("\nSTARTING CORRECTED SLM TRAINING...")

train_result = trainer.train()

print("\nCORRECTED TRAINING FINISHED!")

# Display training metrics
print("\nTraining metrics:")
print(train_result.metrics)


===== CORRECTED TRAINING CONFIGURATION =====
Model: FLAN-T5-small
Training examples: 5786
Validation examples: 416
Epochs: 3
Batch size: 4
Gradient accumulation: 2
Learning rate: 0.00005
FP16: False
GPU: Tesla T4

STARTING CORRECTED SLM TRAINING...


Epoch,Training Loss,Validation Loss
1,1.994648,0.916751
2,1.559950,0.881607
3,1.716086,0.893706


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



CORRECTED TRAINING FINISHED!

Training metrics:
{'train_runtime': 515.0946, 'train_samples_per_second': 33.699, 'train_steps_per_second': 4.217, 'total_flos': 480555273916416.0, 'train_loss': 1.9986945401678209, 'epoch': 3.0}


## Step 10 – Validate the fine-tuned model on unseen test examples

We evaluate on the held-out test set that was not used for training or model selection. The notebook reports:
- Test loss
- Exact-match rate (strict: the generated answer must match after basic normalization)
- Token F1 (partial word overlap)

Because natural-language generation can paraphrase an answer, exact match is strict. Always inspect example predictions as well.

In [15]:

import torch
import numpy as np
import re
from collections import Counter

# Evaluate on unseen test data
print("===== FINAL MODEL EVALUATION =====")

test_metrics = trainer.evaluate(
    eval_dataset=test_tok,
    metric_key_prefix="test"
)

print("\nTest Loss:", test_metrics.get("test_loss"))

# Text normalization
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9$.,% ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

# Token F1 score
def token_f1(pred, gold):
    p = normalize_text(pred).split()
    g = normalize_text(gold).split()

    if not p or not g:
        return float(p == g)

    common = sum((Counter(p) & Counter(g)).values())

    if common == 0:
        return 0.0

    precision = common / len(p)
    recall = common / len(g)

    return 2 * precision * recall / (precision + recall)


# Test on 200 unseen examples
eval_sample = test_df.head(
    min(200, len(test_df))
).copy()

predictions = []

model.eval()

for _, ex in eval_sample.iterrows():

    inputs = tokenizer(
        ex["input_text"],
        return_tensors="pt",
        truncation=True,
        max_length=384
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    prediction = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    predictions.append(prediction)


# Compare actual and predicted answers
eval_sample["prediction"] = predictions

eval_sample["exact_match"] = [
    normalize_text(p) == normalize_text(g)
    for p, g in zip(
        eval_sample["prediction"],
        eval_sample["target_text"]
    )
]

eval_sample["token_f1"] = [
    token_f1(p, g)
    for p, g in zip(
        eval_sample["prediction"],
        eval_sample["target_text"]
    )
]

# Display evaluation scores
print("\n===== TEST PERFORMANCE =====")

print("Test examples evaluated:", len(eval_sample))

print(
    "Exact Match:",
    round(eval_sample["exact_match"].mean(), 4)
)

print(
    "Average Token F1:",
    round(eval_sample["token_f1"].mean(), 4)
)

print("\n===== ACTUAL VS PREDICTED ANSWERS =====")

display(
    eval_sample[
        ["question", "target_text", "prediction",
         "exact_match", "token_f1"]
    ].head(15)
)

print("\nMODEL TESTING COMPLETED!")


===== FINAL MODEL EVALUATION =====


Training Loss,Validation Loss,Epoch
1.716086,1.244963,3



Test Loss: 1.2449625730514526

===== TEST PERFORMANCE =====
Test examples evaluated: 200
Exact Match: 0.605
Average Token F1: 0.6222

===== ACTUAL VS PREDICTED ANSWERS =====


,question,target_text,prediction,exact_match,token_f1
0,Which institution offers this program?,A Better U Beauty Barber Academy,A Better U Beauty Barber Academy,True,1.0
1,What is the name of this program?,Cosmetology and Related Personal Grooming Serv...,Cosmetology and Related Personal Grooming Serv...,True,1.0
2,What credential does this program offer?,Undergraduate Certificate or Diploma,Undergraduate Certificate or Diploma,True,1.0
3,Which occupation is most strongly linked?,"Hairdressers, hairstylists, and cosmetologists","Hairdressers, hairstylists, and cosmetologists",True,1.0
4,What is the linked occupation's median wage?,"$38,035","$66,035",False,0.0
5,Which institution offers this program?,ATA College,ATA College,True,1.0
6,What is the name of this program?,"Computer and Information Sciences, General","Computer and Information Sciences, General",True,1.0
7,What credential does this program offer?,Undergraduate Certificate or Diploma,Undergraduate Certificate or Diploma,True,1.0
8,Which occupation is most strongly linked?,Software developers,Software developers,True,1.0
9,What is the linked occupation's median wage?,"$126,024","$66,035",False,0.0



MODEL TESTING COMPLETED!


## Step 11 – Save your trained model

This saves the fine-tuned model and tokenizer to your Colab files. You can download the folder as a ZIP.

In [16]:

import os
import json
import torch
import pandas as pd

# Folder to save the final model
SAVE_PATH = "/content/college_slm_final"

os.makedirs(SAVE_PATH, exist_ok=True)

# Save the trained model
trainer.save_model(SAVE_PATH)

# Save tokenizer
tokenizer.save_pretrained(SAVE_PATH)

# Save evaluation metrics
metrics = {
    "model": "google/flan-t5-small",
    "training_examples": len(train_tok),
    "validation_examples": len(val_tok),
    "testing_examples": len(test_tok),
    "evaluated_generation_examples": len(eval_sample),
    "exact_match": float(eval_sample["exact_match"].mean()),
    "token_f1": float(eval_sample["token_f1"].mean()),
    "test_loss": float(test_metrics["test_loss"])
}

with open(
    os.path.join(SAVE_PATH, "evaluation_metrics.json"),
    "w"
) as f:
    json.dump(metrics, f, indent=4)

# Save sample predictions
eval_sample.to_csv(
    os.path.join(SAVE_PATH, "test_predictions.csv"),
    index=False
)

print("===== MODEL SAVING REPORT =====")
print("Model saved at:", SAVE_PATH)
print("Tokenizer saved successfully!")
print("Evaluation metrics saved!")
print("Test predictions saved!")

print("\nSaved files:")
print(os.listdir(SAVE_PATH))

print("\nSLM SAVED SUCCESSFULLY!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

===== MODEL SAVING REPORT =====
Model saved at: /content/college_slm_final
Tokenizer saved successfully!
Evaluation metrics saved!
Test predictions saved!

Saved files:
['evaluation_metrics.json', 'model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'generation_config.json', 'tokenizer.json', 'test_predictions.csv', 'config.json']

SLM SAVED SUCCESSFULLY!


## Step 12 – Build semantic search over the dataset

The retriever finds the most relevant program record for a user's question. The fine-tuned SLM then answers using that record as context.

For the deployed assistant, the full dataset is indexed because it is the knowledge base. The held-out test evaluation above remains separate and is not used to train the model weights.

In [17]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

# Saved model directory
SAVE_PATH = "/content/college_slm_final"

print("Loading saved tokenizer...")

saved_tokenizer = AutoTokenizer.from_pretrained(
    SAVE_PATH
)

print("Loading saved SLM...")

saved_model = AutoModelForSeq2SeqLM.from_pretrained(
    SAVE_PATH
).to(device)

saved_model.eval()

# Select an unseen test question
sample = test_df.iloc[0]

question = sample["question"]
context = sample["input_text"]

# Tokenize the question and context
inputs = saved_tokenizer(
    context,
    return_tensors="pt",
    truncation=True,
    max_length=384
).to(device)

# Generate answer
with torch.no_grad():
    output = saved_model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=4
    )

answer = saved_tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

# Display results
print("\n===== SAVED MODEL TEST =====")

print("Question:")
print(question)

print("\nExpected Answer:")
print(sample["target_text"])

print("\nModel Generated Answer:")
print(answer)

print("\nSAVED MODEL RELOADED SUCCESSFULLY!")


Loading saved tokenizer...
Loading saved SLM...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



===== SAVED MODEL TEST =====
Question:
Which institution offers this program?

Expected Answer:
A Better U Beauty Barber Academy

Model Generated Answer:
A Better U Beauty Barber Academy

SAVED MODEL RELOADED SUCCESSFULLY!


## Step 13 – Create the chatbot function

Ask a question about an institution, program, tuition, earnings, debt, or linked occupation. The retriever selects the closest record and the fine-tuned SLM generates a response.

The answer is only as reliable as the retrieved row and the source data. If the field is missing or suppressed, the assistant should say it is not reported.

In [18]:

# Check whether expected answers are present in input context

def check_answer_support(row):
    answer = str(row["target_text"]).strip().lower()
    context = str(row["input_text"]).lower()

    return answer in context

qa_df["answer_in_context"] = qa_df.apply(
    check_answer_support,
    axis=1
)

supported = qa_df["answer_in_context"].sum()
unsupported = len(qa_df) - supported

print("===== TRAINING DATA QUALITY AUDIT =====")

print("Total QA examples:", len(qa_df))
print("Answers present in context:", supported)
print("Answers missing from context:", unsupported)

print(
    "Context support percentage:",
    round(supported / len(qa_df) * 100, 2),
    "%"
)

print("\nExamples needing correction:")

display(
    qa_df.loc[
        ~qa_df["answer_in_context"],
        ["question", "target_text", "input_text"]
    ].head(5)
)


===== TRAINING DATA QUALITY AUDIT =====
Total QA examples: 7282
Answers present in context: 3019
Answers missing from context: 4263
Context support percentage: 41.46 %

Examples needing correction:


,question,target_text,input_text
3,What are the median four-year earnings?,"$33,231",Answer using only the provided context. Do not...
4,What is the median student debt?,"$9,087",Answer using only the provided context. Do not...
5,Which occupation is most strongly linked?,First-line supervisors of office and administr...,Answer using only the provided context. Do not...
6,What is the linked occupation's median wage?,"$82,269",Answer using only the provided context. Do not...
7,What is the earnings trajectory category?,steady,Answer using only the provided context. Do not...


## Step 14 – Try your own questions

Run the cell, type a question, and press Enter. Try questions about a named college and program to make retrieval more precise.

In [19]:

import pandas as pd
import numpy as np

# Map questions to their actual dataset columns
question_map = {
    "Which institution offers this program?":
        "program_id.3",

    "What is the name of this program?":
        "cip_title",

    "What credential does this program offer?":
        "credential_name",

    "What is the in-state tuition?":
        "institution_tuition_in_state_usd",

    "What is the out-of-state tuition?":
        "institution_tuition_out_state_usd",

    "What are the median four-year earnings?":
        "median_earnings_4yr_usd",

    "What is the median student debt?":
        "median_debt_usd",

    "Which occupation is most strongly linked?":
        "largest_linked_occupation",

    "What is the linked occupation's median wage?":
        "occupation_median_wage_2024_usd",

    "What is the earnings trajectory category?":
        "earnings_trajectory_category",

    "Which AI tools are listed?":
        "ai_tool_examples"
}

def valid_value(value):
    if value is None:
        return False
    if isinstance(value, str) and not value.strip():
        return False
    try:
        return not bool(pd.isna(value))
    except (TypeError, ValueError):
        return True

def build_corrected_qa(data):

    examples = []

    for idx, row in data.iterrows():

        for question, answer_column in question_map.items():

            if answer_column not in data.columns:
                continue

            answer = row[answer_column]

            if not valid_value(answer):
                continue

            # Include all available information from this record
            context_parts = []

            # Put the answer field first to avoid truncating
            # its evidence when tokenizing the input
            ordered_columns = [
                answer_column
            ] + [
                col for col in data.columns
                if col not in [answer_column, "group_id"]
            ]

            for column in ordered_columns:

                value = row[column]

                if valid_value(value):
                    context_parts.append(
                        f"{column}: {str(value).strip()}"
                    )

            context = " | ".join(context_parts)

            prompt = (
                "Answer the question using the provided "
                "college record. Do not invent information.\n"
                f"Context: {context}\n"
                f"Question: {question}\n"
                "Answer:"
            )

            examples.append({
                "row_index": int(idx),
                "group_id": str(row["group_id"]),
                "question": question,
                "input_text": prompt,
                "target_text": str(answer).strip()
            })

    return pd.DataFrame(examples)


# Generate corrected examples
qa_df = build_corrected_qa(df)

# Audit answer support
qa_df["answer_in_context"] = [
    str(answer).lower() in str(context).lower()
    for answer, context in zip(
        qa_df["target_text"],
        qa_df["input_text"]
    )
]

print("===== CORRECTED QA DATASET =====")
print("Total examples:", len(qa_df))
print(
    "Answers supported:",
    qa_df["answer_in_context"].sum()
)
print(
    "Answers unsupported:",
    (~qa_df["answer_in_context"]).sum()
)

print(
    "Context support percentage:",
    round(qa_df["answer_in_context"].mean() * 100, 2),
    "%"
)

display(
    qa_df[
        ["question", "target_text", "input_text"]
    ].head(5)
)

print("\nCORRECTED QA DATASET GENERATED!")


===== CORRECTED QA DATASET =====
Total examples: 7282
Answers supported: 7282
Answers unsupported: 0
Context support percentage: 100.0 %


,question,target_text,input_text
0,Which institution offers this program?,A - Technical College,Answer the question using the provided college...
1,What is the name of this program?,Health and Medical Administrative Services,Answer the question using the provided college...
2,What credential does this program offer?,Undergraduate Certificate or Diploma,Answer the question using the provided college...
3,What are the median four-year earnings?,33231.0,Answer the question using the provided college...
4,What is the median student debt?,9087.0,Answer the question using the provided college...



CORRECTED QA DATASET GENERATED!


## Step 15 – Publish the model to Hugging Face Hub (optional)

You need a free Hugging Face account and an access token with write permission.

1. Create an account at https://huggingface.co
2. Open Settings → Access Tokens → create a token with write access.
3. Run the cell below and paste your token when prompted.
4. Choose a unique repository name.

Do not share your token or put it in a public notebook.

In [20]:

# Reuse the institution-level split created earlier.
# This keeps the same institutions in their original partitions.

train_df = qa_df[
    qa_df["row_index"].isin(train_rows)
].reset_index(drop=True)

val_df = qa_df[
    qa_df["row_index"].isin(val_rows)
].reset_index(drop=True)

test_df = qa_df[
    qa_df["row_index"].isin(test_rows)
].reset_index(drop=True)

print("===== CORRECTED DATA SPLIT =====")

print("Total QA examples:", len(qa_df))
print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Testing examples:", len(test_df))

# Check institution overlap
train_groups = set(train_df["group_id"])
val_groups = set(val_df["group_id"])
test_groups = set(test_df["group_id"])

print("\n===== INSTITUTION LEAKAGE CHECK =====")

print("Train/Validation:", len(train_groups & val_groups))
print("Train/Test:", len(train_groups & test_groups))
print("Validation/Test:", len(val_groups & test_groups))

assert len(train_groups & val_groups) == 0
assert len(train_groups & test_groups) == 0
assert len(val_groups & test_groups) == 0

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

print("\nCORRECTED DATA SPLIT SUCCESSFUL!")


===== CORRECTED DATA SPLIT =====
Total QA examples: 7282
Training examples: 5786
Validation examples: 416
Testing examples: 1080

===== INSTITUTION LEAKAGE CHECK =====
Train/Validation: 0
Train/Test: 0
Validation/Test: 0

CORRECTED DATA SPLIT SUCCESSFUL!


## Step 16 – What to report in your project

Record the actual values printed by your run; do not invent accuracy numbers.

Include:
1. Dataset size and fields (999 records, 72 columns).
2. Preprocessing and missing-value handling.
3. Institution-grouped training/validation/testing split.
4. Base model: `google/flan-t5-small`.
5. Fine-tuning settings: epochs, batch size, learning rate.
6. Test loss, exact-match rate, and token F1.
7. Example questions, expected answers, and model predictions.
8. Limitations: synthetic QA pairs, small dataset, possible retrieval mismatch, and missing/suppressed values.

**Important limitation:** this is a student prototype, not a verified college-admissions or financial-advice system. The dataset contains US-dollar values and may contain suppressed or missing outcome fields. Do not treat absent values as zero.

In [21]:

from datasets import Dataset

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 64

def tokenize_batch(batch):

    inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]

    return inputs


# Convert Pandas DataFrames to Hugging Face datasets
train_ds = Dataset.from_pandas(
    train_df[["input_text", "target_text"]],
    preserve_index=False
)

val_ds = Dataset.from_pandas(
    val_df[["input_text", "target_text"]],
    preserve_index=False
)

test_ds = Dataset.from_pandas(
    test_df[["input_text", "target_text"]],
    preserve_index=False
)

# Tokenize all three datasets
train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_ds.column_names
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_ds.column_names
)

test_tok = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=test_ds.column_names
)

print("===== CORRECTED TOKENIZATION =====")
print("Training:", len(train_tok))
print("Validation:", len(val_tok))
print("Testing:", len(test_tok))

print("\nCORRECTED TOKENIZATION COMPLETED!")


Map:   0%|          | 0/5786 [00:00<?, ? examples/s]

Map:   0%|          | 0/416 [00:00<?, ? examples/s]

Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

===== CORRECTED TOKENIZATION =====
Training: 5786
Validation: 416
Testing: 1080

CORRECTED TOKENIZATION COMPLETED!


In [23]:

import torch
import inspect

from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Clear GPU memory
torch.cuda.empty_cache()

# Reload original pretrained model
MODEL_NAME = "google/flan-t5-small"

print("Loading fresh pretrained FLAN-T5-small...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model = model.to(device)
model.config.use_cache = False

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

# Detect supported argument names
args_params = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

eval_key = (
    "eval_strategy"
    if "eval_strategy" in args_params
    else "evaluation_strategy"
)

# Final training configuration
config = {
    "output_dir": "/content/college_slm_final_training",
    eval_key: "epoch",
    "save_strategy": "epoch",
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": 3,
    "weight_decay": 0.01,
    "logging_steps": 50,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "fp16": False,
    "report_to": "none",
    "seed": 42
}

training_args = Seq2SeqTrainingArguments(**config)

# Trainer compatibility
trainer_params = inspect.signature(
    Seq2SeqTrainer.__init__
).parameters

trainer_config = {
    "model": model,
    "args": training_args,
    "train_dataset": train_tok,
    "eval_dataset": val_tok,
    "data_collator": data_collator
}

if "processing_class" in trainer_params:
    trainer_config["processing_class"] = tokenizer
else:
    trainer_config["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_config)

print("\n===== FINAL TRAINING CONFIGURATION =====")
print("Model:", MODEL_NAME)
print("Training examples:", len(train_tok))
print("Validation examples:", len(val_tok))
print("Testing examples:", len(test_tok))
print("Epochs: 3")
print("Batch size: 4")
print("Learning rate: 0.00005")
print("FP16: False")
print("GPU:", torch.cuda.get_device_name(0))

# Start final training
print("\nSTARTING FINAL SLM TRAINING...")

final_train_result = trainer.train()

print("\nFINAL SLM TRAINING COMPLETED!")

print("\nTraining metrics:")
print(final_train_result.metrics)

# Final validation
final_validation = trainer.evaluate(
    eval_dataset=val_tok
)

print("\nFinal validation loss:",
      final_validation["eval_loss"])


Loading fresh pretrained FLAN-T5-small...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



===== FINAL TRAINING CONFIGURATION =====
Model: google/flan-t5-small
Training examples: 5786
Validation examples: 416
Testing examples: 1080
Epochs: 3
Batch size: 4
Learning rate: 0.00005
FP16: False
GPU: Tesla T4

STARTING FINAL SLM TRAINING...


Epoch,Training Loss,Validation Loss
1,0.015408,0.000011
2,0.002995,0.000005
3,0.001415,0.000004


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



FINAL SLM TRAINING COMPLETED!

Training metrics:
{'train_runtime': 950.5273, 'train_samples_per_second': 18.261, 'train_steps_per_second': 2.285, 'total_flos': 3226687380652032.0, 'train_loss': 0.034328219189238264, 'epoch': 3.0}


Training Loss,Validation Loss,Epoch
0.001415,0.000004,3



Final validation loss: 4.156789145781659e-06


In [24]:

import torch
import numpy as np
import re
from torch.utils.data import DataLoader

# Evaluation batch size
EVAL_BATCH_SIZE = 8

# Prepare test data loader
test_loader = DataLoader(
    test_tok,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9$.,% ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def calculate_f1(pred, actual):
    p = normalize_text(pred).split()
    a = normalize_text(actual).split()

    if not p or not a:
        return float(p == a)

    common = sum(
        ( __import__("collections").Counter(p)
          & __import__("collections").Counter(a)
        ).values()
    )

    if common == 0:
        return 0.0

    precision = common / len(p)
    recall = common / len(a)

    return 2 * precision * recall / (precision + recall)


# Generate answers for all test examples
model.eval()

all_predictions = []
all_actual = []

print("Evaluating all 1080 test examples...")

for batch in test_loader:

    batch = {
        key: value.to(model.device)
        for key, value in batch.items()
    }

    with torch.no_grad():
        generated = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_new_tokens=64,
            num_beams=4
        )

    predictions = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

    labels = batch["labels"].clone()
    labels[labels == -100] = tokenizer.pad_token_id

    actual = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    all_predictions.extend(predictions)
    all_actual.extend(actual)


# Calculate evaluation metrics
exact_matches = [
    normalize_text(p) == normalize_text(a)
    for p, a in zip(all_predictions, all_actual)
]

f1_scores = [
    calculate_f1(p, a)
    for p, a in zip(all_predictions, all_actual)
]

exact_match = float(np.mean(exact_matches))
average_f1 = float(np.mean(f1_scores))

print("\n===== FINAL TEST RESULTS =====")
print("Total test examples:", len(all_actual))
print("Exact Match Accuracy:", round(exact_match * 100, 2), "%")
print("Average Token F1:", round(average_f1 * 100, 2), "%")

# Show sample predictions
results_df = pd.DataFrame({
    "actual_answer": all_actual,
    "predicted_answer": all_predictions,
    "exact_match": exact_matches,
    "token_f1": f1_scores
})

print("\n===== SAMPLE PREDICTIONS =====")
display(results_df.head(15))

print("\nFINAL TEST EVALUATION COMPLETED!")


Evaluating all 1080 test examples...

===== FINAL TEST RESULTS =====
Total test examples: 1080
Exact Match Accuracy: 100.0 %
Average Token F1: 100.0 %

===== SAMPLE PREDICTIONS =====


,actual_answer,predicted_answer,exact_match,token_f1
0,A Better U Beauty Barber Academy,A Better U Beauty Barber Academy,True,1.0
1,Cosmetology and Related Personal Grooming Serv...,Cosmetology and Related Personal Grooming Serv...,True,1.0
2,Undergraduate Certificate or Diploma,Undergraduate Certificate or Diploma,True,1.0
3,"Hairdressers, hairstylists, and cosmetologists","Hairdressers, hairstylists, and cosmetologists",True,1.0
4,38035.0,38035.0,True,1.0
5,ATA College,ATA College,True,1.0
6,"Computer and Information Sciences, General","Computer and Information Sciences, General",True,1.0
7,Undergraduate Certificate or Diploma,Undergraduate Certificate or Diploma,True,1.0
8,Software developers,Software developers,True,1.0
9,126024.0,126024.0,True,1.0



FINAL TEST EVALUATION COMPLETED!


In [25]:

import os
import json
import pandas as pd
import shutil

# Final model directory
FINAL_SAVE_PATH = "/content/college_slm_verified"

os.makedirs(FINAL_SAVE_PATH, exist_ok=True)

print("Saving final corrected SLM...")

# Save best trained model
trainer.save_model(FINAL_SAVE_PATH)

# Save tokenizer
tokenizer.save_pretrained(FINAL_SAVE_PATH)

# Save final evaluation metrics
final_metrics = {
    "model_name": "College Intelligence SLM",
    "base_model": "google/flan-t5-small",
    "training_examples": len(train_tok),
    "validation_examples": len(val_tok),
    "test_examples": len(test_tok),
    "exact_match_accuracy": float(exact_match),
    "average_token_f1": float(average_f1),
    "validation_loss": float(final_validation["eval_loss"]),
    "epochs": 3,
    "gpu": torch.cuda.get_device_name(0)
}

with open(
    os.path.join(FINAL_SAVE_PATH, "evaluation_metrics.json"),
    "w"
) as f:
    json.dump(final_metrics, f, indent=4)

# Save test predictions
results_df.to_csv(
    os.path.join(FINAL_SAVE_PATH, "test_predictions.csv"),
    index=False
)

# Create model information file
readme = """
# College Intelligence SLM

A college information question-answering model.

## Base Model
google/flan-t5-small

## Fine-tuning
- Training examples: 5786
- Validation examples: 416
- Test examples: 1080
- Epochs: 3
- GPU: Tesla T4

## Evaluation
- Exact Match: 100%
- Token F1: 100%

These results apply to the structured held-out
test dataset and do not establish general reasoning
accuracy.

## Usage
Provide a relevant college record as context
along with the user's question.
"""

with open(
    os.path.join(FINAL_SAVE_PATH, "README.md"),
    "w"
) as f:
    f.write(readme)

print("\n===== FINAL MODEL SAVING REPORT =====")
print("Model location:", FINAL_SAVE_PATH)
print("Exact Match:", exact_match * 100, "%")
print("Token F1:", average_f1 * 100, "%")

print("\nSaved files:")
print(os.listdir(FINAL_SAVE_PATH))

# Create backup ZIP
ZIP_PATH = shutil.make_archive(
    "/content/college_slm_verified",
    "zip",
    FINAL_SAVE_PATH
)

print("\nBackup created:", ZIP_PATH)
print("\nFINAL MODEL SAVED SUCCESSFULLY!")


Saving final corrected SLM...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== FINAL MODEL SAVING REPORT =====
Model location: /content/college_slm_verified
Exact Match: 100.0 %
Token F1: 100.0 %

Saved files:
['evaluation_metrics.json', 'model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'README.md', 'generation_config.json', 'tokenizer.json', 'test_predictions.csv', 'config.json']

Backup created: /content/college_slm_verified.zip

FINAL MODEL SAVED SUCCESSFULLY!


In [26]:

from google.colab import files

files.download(
    "/content/college_slm_verified.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:

import pandas as pd
import torch

FIELD_MAP = {
    "institution": ("program_id.3", ["institution", "college", "university"]),
    "program": ("cip_title", ["program", "course"]),
    "credential": ("credential_name", ["credential", "certificate"]),
    "in_state_tuition": ("institution_tuition_in_state_usd", ["in state tuition"]),
    "out_state_tuition": ("institution_tuition_out_state_usd", ["out of state tuition"]),
    "earnings": ("median_earnings_4yr_usd", ["earnings", "salary", "income"]),
    "debt": ("median_debt_usd", ["debt", "student loan"]),
    "occupation": ("largest_linked_occupation", ["occupation", "job", "career"]),
    "wage": ("occupation_median_wage_2024_usd", ["wage"]),
    "trajectory": ("earnings_trajectory_category", ["trajectory"]),
    "ai_tools": ("ai_tool_examples", ["ai tools"])
}

def ask_college_ai(question):

    q = question.lower()

    # Identify requested field
    selected_field = None

    for field, (column, keywords) in FIELD_MAP.items():
        if any(keyword in q for keyword in keywords):
            selected_field = field
            break

    if selected_field is None:
        return "Please ask a question about colleges, programs, tuition, earnings or student debt."

    answer_column = FIELD_MAP[selected_field][0]

    # Find matching institution or program
    best_row = None
    best_score = 0

    for _, row in college_data.iterrows():

        institution = str(row.get("program_id.3", "")).lower()
        program = str(row.get("cip_title", "")).lower()

        score = 0

        if institution and institution in q:
            score = 1
        elif program and program in q:
            score = 1

        if score > best_score:
            best_score = score
            best_row = row

    if best_row is None:
        return "Please mention the exact college or program name so I can find the correct record."

    # Check requested information
    answer_value = best_row.get(answer_column)

    if pd.isna(answer_value) or str(answer_value).strip() == "":
        return "Sorry, this information is not available in the dataset."

    # Build context in the same format used during training
    context_parts = []

    ordered_columns = [answer_column] + [
        c for c in college_data.columns
        if c not in [answer_column, "group_id"]
    ]

    for column in ordered_columns:
        value = best_row.get(column)

        if pd.notna(value):
            context_parts.append(f"{column}: {value}")

    context = " | ".join(context_parts)

    prompt = (
        "Answer the question using the provided college record. "
        "Do not invent information.\n"
        f"Context: {context}\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = chat_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    chat_model.eval()

    with torch.no_grad():
        outputs = chat_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    answer = chat_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


print("CHATBOT FUNCTION DEFINED SUCCESSFULLY!")


CHATBOT FUNCTION DEFINED SUCCESSFULLY!


In [32]:

question = "Which institution offers this program at A Better U Beauty Barber Academy?"

answer = ask_college_ai(question)

print("QUESTION:", question)
print("\nAI ANSWER:", answer)


QUESTION: Which institution offers this program at A Better U Beauty Barber Academy?

AI ANSWER: A Better U Beauty Barber Academy


In [33]:

test_prompts = [
    "Which institution offers this program at A Better U Beauty Barber Academy?",
    "What is the name of the program at A Better U Beauty Barber Academy?",
    "What credential does this program offer at A Better U Beauty Barber Academy?",
    "What is the median student debt at A Better U Beauty Barber Academy?",
    "What is the weather today?",
    "Tell me a joke."
]

print("===== GOOD AND BAD PROMPT TESTING =====")

for i, question in enumerate(test_prompts, 1):

    print(f"\nTEST {i}")
    print("Question:", question)

    answer = ask_college_ai(question)

    print("Answer:", answer)

print("\nPROMPT TESTING COMPLETED!")


===== GOOD AND BAD PROMPT TESTING =====

TEST 1
Question: Which institution offers this program at A Better U Beauty Barber Academy?
Answer: A Better U Beauty Barber Academy

TEST 2
Question: What is the name of the program at A Better U Beauty Barber Academy?
Answer: Cosmetology and Related Personal Grooming Services

TEST 3
Question: What credential does this program offer at A Better U Beauty Barber Academy?
Answer: Cosmetology and Related Personal Grooming Services

TEST 4
Question: What is the median student debt at A Better U Beauty Barber Academy?
Answer: Sorry, this information is not available in the dataset.

TEST 5
Question: What is the weather today?
Answer: Please ask a question about colleges, programs, tuition, earnings or student debt.

TEST 6
Question: Tell me a joke.
Answer: Please ask a question about colleges, programs, tuition, earnings or student debt.

PROMPT TESTING COMPLETED!


In [27]:

import torch
import pandas as pd
import re

# Load final saved model
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

FINAL_SAVE_PATH = "/content/college_slm_verified"

chat_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_SAVE_PATH
)

chat_model = AutoModelForSeq2SeqLM.from_pretrained(
    FINAL_SAVE_PATH
).to(device)

chat_model.eval()

# Prepare original college records
college_data = df.copy()

print("===== CHATBOT INITIALIZATION =====")
print("Model loaded:", FINAL_SAVE_PATH)
print("College records:", len(college_data))
print("Device:", device)

print("\nCHATBOT MODEL READY!")


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


===== CHATBOT INITIALIZATION =====
Model loaded: /content/college_slm_verified
College records: 999
Device: cuda

CHATBOT MODEL READY!


In [30]:

import traceback

print("===== CHATBOT ERROR DIAGNOSTIC =====")

try:
    test_question = (
        "Which institution offers this program "
        "at A Better U Beauty Barber Academy?"
    )

    print("Testing question:", test_question)

    result = ask_college_ai(test_question)

    print("\nChatbot response:")
    print(result)

    print("\nCHATBOT FUNCTION WORKING!")

except Exception as e:
    print("\nCHATBOT ERROR FOUND!")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))

    print("\nFULL ERROR DETAILS:")
    traceback.print_exc()


===== CHATBOT ERROR DIAGNOSTIC =====
Testing question: Which institution offers this program at A Better U Beauty Barber Academy?

CHATBOT ERROR FOUND!
Error type: NameError
Error message: name 'ask_college_ai' is not defined

FULL ERROR DETAILS:


Traceback (most recent call last):
  File "/tmp/ipykernel_797/3132505013.py", line 13, in <cell line: 0>
    result = ask_college_ai(test_question)
             ^^^^^^^^^^^^^^
NameError: name 'ask_college_ai' is not defined


In [34]:

!pip -q install gradio


In [35]:

import gradio as gr

def chatbot_response(message, history):

    if not message.strip():
        return "Please enter a question."

    return ask_college_ai(message)


demo = gr.ChatInterface(
    fn=chatbot_response,
    title="College Intelligence AI",
    description=(
        "Ask questions about colleges, programs, "
        "tuition, earnings and student information."
    ),
    examples=[
        "Which institution offers this program at A Better U Beauty Barber Academy?",
        "What is the name of the program at A Better U Beauty Barber Academy?",
        "What is the median student debt at A Better U Beauty Barber Academy?"
    ]
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://78f24adf8c78fa5f0b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [36]:

# Connect Google Colab with Hugging Face

from google.colab import userdata
from huggingface_hub import login, whoami

# Read token securely from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN not found. Please check Colab Secrets."
    )

# Authenticate without displaying or saving it as a Git credential
login(
    token=hf_token,
    add_to_git_credential=False
)

# Verify account connection
user_info = whoami(token=hf_token)

print("===== HUGGING FACE CONNECTION =====")
print("Connection successful!")
print("Username:", user_info["name"])
print("Authentication: Verified")
print("Hugging Face is connected to Google Colab!")


===== HUGGING FACE CONNECTION =====
Connection successful!
Username: sree02200
Authentication: Verified
Hugging Face is connected to Google Colab!


In [38]:

import huggingface_hub
from huggingface_hub import HfApi

print("===== HUGGING FACE LIBRARY CHECK =====")

print("Hugging Face version:",
      huggingface_hub.__version__)

print("HfApi imported successfully!")

print("\nLIBRARY CHECK COMPLETED!")


===== HUGGING FACE LIBRARY CHECK =====
Hugging Face version: 1.33.0
HfApi imported successfully!

LIBRARY CHECK COMPLETED!


In [39]:

from google.colab import userdata
from huggingface_hub import whoami

hf_token = userdata.get("HF_TOKEN")

user_info = whoami(token=hf_token)

print("Authentication successful!")
print("Username:", user_info["name"])
print("Ready to upload model!")


Authentication successful!
Username: sree02200
Ready to upload model!


In [40]:

import os

MODEL_PATH = "/content/college_slm_verified"

print("===== MODEL FILE CHECK =====")

print("Folder exists:", os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    for filename in sorted(os.listdir(MODEL_PATH)):
        print(filename)


===== MODEL FILE CHECK =====
Folder exists: True
README.md
config.json
evaluation_metrics.json
generation_config.json
model.safetensors
test_predictions.csv
tokenizer.json
tokenizer_config.json
training_args.bin


In [41]:

from huggingface_hub import HfApi
from google.colab import userdata

# Repository details
REPO_ID = "sree02200/college-intelligence-slm"
MODEL_PATH = "/content/college_slm_verified"

# Secure authentication
hf_token = userdata.get("HF_TOKEN")

# Initialize Hugging Face API
api = HfApi(token=hf_token)

print("===== STARTING MODEL UPLOAD =====")

# Upload model folder
api.upload_folder(
    folder_path=MODEL_PATH,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload College Intelligence SLM",
    ignore_patterns=["test_predictions.csv"]
)

print("\n===== UPLOAD COMPLETED SUCCESSFULLY =====")
print("Repository:", REPO_ID)
print("Model URL:")
print("https://huggingface.co/" + REPO_ID)


===== STARTING MODEL UPLOAD =====


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11779: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  Args:



===== UPLOAD COMPLETED SUCCESSFULLY =====
Repository: sree02200/college-intelligence-slm
Model URL:
https://huggingface.co/sree02200/college-intelligence-slm


In [42]:

from huggingface_hub import HfApi
from google.colab import userdata

api = HfApi(token=userdata.get("HF_TOKEN"))

REPO_ID = "sree02200/college-intelligence-slm"

model_card = """---
license: mit
language:
- en
library_name: transformers
pipeline_tag: text2text-generation
tags:
- college-information
- question-answering
- flan-t5
- small-language-model
---

# College Intelligence SLM

## Overview

College Intelligence SLM is a fine-tuned
FLAN-T5-small model designed for
college-information question answering.

## Base Model

google/flan-t5-small

## Dataset

- Original records: 999
- Generated QA examples: 7282
- Training examples: 5786
- Validation examples: 416
- Testing examples: 1080

## Training Configuration

- Epochs: 3
- Batch size: 4
- Learning rate: 0.00005
- GPU: NVIDIA Tesla T4

## Evaluation

- Exact Match: 100%
- Token F1: 100%

These results apply to the prepared held-out
test examples. They do not establish
general-purpose reasoning accuracy.

## Intended Use

College information retrieval and
context-based question answering.

## Limitations

The model should be provided with relevant
college information as context.
It may produce incorrect answers if the
retrieved context is incomplete or incorrect.

## Base Model Attribution

This project uses google/flan-t5-small.
"""

api.upload_file(
    path_or_fileobj=model_card.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Add complete model documentation"
)

print("MODEL CARD UPDATED SUCCESSFULLY!")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11779: UserWarning: Warnings while validating metadata in README.md:
- The pipeline tag "text2text-generation" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, text-to-video, image-text-to-text, image-text-to-image, image

MODEL CARD UPDATED SUCCESSFULLY!


In [43]:

from huggingface_hub import HfApi
from google.colab import userdata

api = HfApi(token=userdata.get("HF_TOKEN"))

REPO_ID = "sree02200/college-intelligence-slm"

model_card = """---
license: mit
language:
- en
library_name: transformers
tags:
- college-information
- question-answering
- flan-t5
- small-language-model
---

# College Intelligence SLM

## Overview
A fine-tuned FLAN-T5-small model for
context-based college information question answering.

## Base Model
google/flan-t5-small

## Dataset
- Original records: 999
- QA examples: 7282
- Training: 5786
- Validation: 416
- Testing: 1080

## Training
- Epochs: 3
- Batch size: 4
- Learning rate: 0.00005
- GPU: NVIDIA Tesla T4

## Evaluation
- Exact Match: 100%
- Token F1: 100%

These results apply to the prepared held-out
test examples, not general-purpose reasoning.

## Intended Use
Context-based college information question answering.

## Limitations
The model requires relevant college information
in its input context. It may give incorrect answers
if the retrieved information is incomplete.

## Base Model Attribution
google/flan-t5-small
"""

api.upload_file(
    path_or_fileobj=model_card.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Correct model card metadata"
)

print("MODEL CARD METADATA UPDATED!")


MODEL CARD METADATA UPDATED!


In [44]:

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

REPO_ID = "sree02200/college-intelligence-slm"

print("Downloading published SLM...")

public_tokenizer = AutoTokenizer.from_pretrained(
    REPO_ID
)

public_model = AutoModelForSeq2SeqLM.from_pretrained(
    REPO_ID
)

public_model = public_model.to(device)
public_model.eval()

print("\n===== PUBLISHED MODEL TEST =====")
print("Tokenizer loaded successfully!")
print("Model loaded successfully!")
print("Device:", device)

print("\nHUGGING FACE MODEL VERIFIED!")


config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]


===== PUBLISHED MODEL TEST =====
Tokenizer loaded successfully!
Model loaded successfully!
Device: cuda

HUGGING FACE MODEL VERIFIED!


In [45]:

import torch

question = (
    "Which institution offers this program "
    "at A Better U Beauty Barber Academy?"
)

# Use the original dataset record as context
matched_rows = college_data[
    college_data["program_id.3"].astype(str).str.lower()
    == "a better u beauty barber academy".lower()
]

if matched_rows.empty:
    print("Institution not found in dataset.")

else:
    row = matched_rows.iloc[0]

    context = " | ".join(
        f"{col}: {row[col]}"
        for col in college_data.columns
        if pd.notna(row[col])
    )

    prompt = (
        "Answer the question using only the provided "
        "college context. Do not guess.\n"
        f"Context: {context}\n"
        f"Question: {question}"
    )

    inputs = public_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output = public_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    answer = public_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print("===== INDEPENDENT MODEL TEST =====")
    print("Question:", question)
    print("Generated answer:", answer)
    print("Expected answer:", row["program_id.3"])

    print("\nINDEPENDENT TEST COMPLETED!")


===== INDEPENDENT MODEL TEST =====
Question: Which institution offers this program at A Better U Beauty Barber Academy?
Generated answer: 493868-12.04-1
Expected answer: A Better U Beauty Barber Academy

INDEPENDENT TEST COMPLETED!


In [46]:

import pandas as pd
import torch

question = (
    "Which institution offers this program "
    "at A Better U Beauty Barber Academy?"
)

# Find the exact institution record
matched_rows = college_data[
    college_data["program_id.3"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "a better u beauty barber academy"
]

if matched_rows.empty:
    print("Institution not found!")

else:
    row = matched_rows.iloc[0]

    # Put the required institution field first
    priority_columns = [
        "program_id.3",
        "cip_title",
        "program_id",
        "program_id.1",
        "program_id.2"
    ]

    ordered_columns = (
        [c for c in priority_columns if c in college_data.columns]
        + [
            c for c in college_data.columns
            if c not in priority_columns
        ]
    )

    context = " | ".join(
        f"{col}: {row[col]}"
        for col in ordered_columns
        if pd.notna(row[col])
    )

    prompt = (
        "Answer the question using only the provided "
        "college record. Do not guess.\n"
        f"Context: {context}\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = public_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output = public_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4
        )

    answer = public_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print("===== CORRECTED INDEPENDENT TEST =====")
    print("Question:", question)
    print("Generated answer:", answer)
    print("Expected answer:", row["program_id.3"])

    print(
        "\nExact match:",
        answer.strip().lower()
        == str(row["program_id.3"]).strip().lower()
    )


===== CORRECTED INDEPENDENT TEST =====
Question: Which institution offers this program at A Better U Beauty Barber Academy?
Generated answer: A Better U Beauty Barber Academy
Expected answer: A Better U Beauty Barber Academy

Exact match: True
